In [8]:
customer = pd.read_csv(CUSTOMER_FILE, encoding="utf-8-sig")
product  = pd.read_csv(PRODUCT_FILE,  encoding="utf-8-sig")
sale     = pd.read_csv(SALES_FILE,    encoding="utf-8-sig")

for df in (customer, product, sale):
    df.columns = df.columns.str.lower()

print(customer.shape, product.shape, sale.shape)   

(40000, 15) (2000, 12) (250000, 21)


In [9]:
customer["phone"]   = customer["phone"].abs().astype(str)
customer["pincode"] = customer["pincode"].astype(str)
customer["state"]   = customer["state"].replace({"UP": "Uttar Pradesh"})

customer["date_of_birth"]     = pd.to_datetime(customer["date_of_birth"],     format="%d-%m-%Y")
customer["registration_date"] = pd.to_datetime(customer["registration_date"], format="%d-%m-%Y")

# flag, don't delete, customers who registered before turning 18
customer["registered_under_18"] = (
    customer["registration_date"] < customer["date_of_birth"] + pd.DateOffset(years=18)
)

In [10]:
brand_fix = {"Peter": "Peter England", "Urban": "Urban Ladder"}
product["brand"] = product["brand"].replace(brand_fix)
for short, full in brand_fix.items():
    product["product_name"] = product["product_name"].str.replace(rf"^{short} ", f"{full} ", regex=True)

# "Samsung Mobile V5" -> "Samsung Mobile"
product["product_line"] = product["product_name"].str.replace(r"\s+V\d+$", "", regex=True)

In [11]:
sale["order_date"]    = pd.to_datetime(sale["order_date"],    format="%d-%m-%Y")
sale["delivery_date"] = pd.to_datetime(sale["delivery_date"], format="%d-%m-%Y")
sale["order_datetime"] = pd.to_datetime(
    sale["order_date"].dt.strftime("%Y-%m-%d") + " " + sale["order_time"]
)

sale["total_amount"] = sale["total_amount"].clip(lower=0)
sale["coupon_code"]  = sale["coupon_code"].fillna("No Coupon")

sale["expected_delivery_date"] = sale["delivery_date"].where(
    sale["order_status"].isin(["Processing", "Shipped"])
)
sale.loc[sale["order_status"].isin(["Cancelled", "Processing", "Shipped"]), "delivery_date"] = pd.NaT
sale["delivery_days"] = (sale["delivery_date"] - sale["order_date"]).dt.days

sale = sale.drop(columns=["city", "state", "customer_age", "customer_age_group"])

In [12]:
rows_before = len(sale)

sale = sale.merge(
    customer[["customer_id", "gender", "city", "state", "age", "age_group", "customer_tier"]],
    on="customer_id", how="left", validate="m:1"
)
sale = sale.merge(
    product[["product_id", "product_name", "product_line", "category", "brand",
             "original_price", "discount_percent"]],
    on="product_id", how="left", validate="m:1"
)
print(rows_before, len(sale))   

250000 250000


In [13]:
assert len(sale) == 250000
assert (sale["total_amount"] < 0).sum() == 0
assert "UP" not in sale["state"].values
assert sale[["city", "state", "age", "category", "brand"]].isna().sum().sum() == 0
assert sale["order_id"].is_unique
print("All checks passed")

customer.to_csv("customer_clean.csv", index=False)
product.to_csv("product_clean.csv",  index=False)
sale.to_csv("sales_clean.csv",       index=False)

All checks passed


In [14]:
sale["order_year"]  = sale["order_date"].dt.year
sale["order_month"] = sale["order_date"].dt.to_period("M").astype(str)
sale["order_hour"]  = sale["order_datetime"].dt.hour

delivered = sale[sale["order_status"] == "Delivered"]   # use this for revenue

In [15]:
sale["order_year"]  = sale["order_date"].dt.year
sale["order_month"] = sale["order_date"].dt.to_period("M").astype(str)
sale["order_hour"]  = sale["order_datetime"].dt.hour
sale["discount_given"] = sale["coupon_discount"] > 0

In [16]:
delivered = sale[sale["order_status"] == "Delivered"]

In [17]:
monthly = delivered.groupby("order_month")["total_amount"].sum()


In [18]:
by_cat  = delivered.groupby("category")["total_amount"].agg(["sum", "count", "mean"])


In [19]:
top_city = delivered.groupby("city")["total_amount"].sum().sort_values(ascending=False).head(10)

In [20]:
sale.groupby("category")["order_status"].apply(lambda s: (s == "Cancelled").mean())

category
Beauty         0.049749
Books          0.049766
Electronics    0.049638
Fashion        0.050661
Grocery        0.048680
Home           0.050278
Sports         0.051430
Name: order_status, dtype: float64

In [21]:
pd.pivot_table(delivered, index="category", columns="order_year",
               values="total_amount", aggfunc="sum")

order_year,2024,2025,2026
category,,,
Beauty,8.366369e+06,1.490552e+07,7.387162e+06
Books,4.207538e+06,7.158815e+06,3.536970e+06
Electronics,9.636868e+08,1.655717e+09,8.238837e+08
Fashion,7.491331e+07,1.277825e+08,6.388360e+07
Grocery,9.164788e+06,1.647772e+07,7.816319e+06
Home,1.587315e+08,2.762371e+08,1.383174e+08
Sports,1.049716e+08,1.850403e+08,8.937985e+07


In [22]:
cust_summary = delivered.groupby("customer_id").agg(
    orders=("order_id", "count"),
    spent=("total_amount", "sum"),
    last_order=("order_date", "max")
)

In [23]:
sale["order_size"] = pd.cut(sale["order_value"], [0, 1000, 10000, 50000, np.inf],
                            labels=["Small", "Medium", "Large", "Very large"])

In [24]:
# Repeat vs one-time customers
orders_per_cust = delivered.groupby("customer_id")["order_id"].count()
(orders_per_cust > 1).mean() * 100          # % who ordered more than once

# Days since each customer's last order (recency)
last = delivered.groupby("customer_id")["order_date"].max()
(delivered["order_date"].max() - last).dt.days

customer_id
CUST00000001    169
CUST00000002    137
CUST00000004    275
CUST00000005    199
CUST00000006    135
               ... 
CUST00039996     36
CUST00039997     90
CUST00039998     35
CUST00039999    256
CUST00040000     40
Name: order_date, Length: 39726, dtype: int64

In [25]:
# Products never sold, or sold the least
sold = delivered.groupby("product_id")["quantity"].sum()
product.set_index("product_id").join(sold).fillna({"quantity": 0}).sort_values("quantity").head(10)

# Does a higher discount sell more?
delivered.groupby("discount_percent")["quantity"].sum()

discount_percent
5     6293
6     5779
7     4909
8     5943
9     5631
10    3971
11    6416
12    3926
13    4684
14    4443
15    6746
16    6217
17    5782
18    5135
19    4260
20    5433
21    6816
22    4466
23    5712
24    4019
25    5571
26    5660
27    6667
28    4517
29    5008
30    6223
31    7110
32    4269
33    4751
34    5830
35    5532
36    5299
37    4804
38    5764
39    4730
40    4634
41    6835
42    4694
43    6856
44    6327
45    5996
46    4379
47    6235
48    5046
49    5421
50    5095
Name: quantity, dtype: int64

In [26]:
delivered.groupby(delivered["order_date"].dt.day_name())["total_amount"].sum()   # best weekday


order_date
Friday       6.638069e+08
Monday       6.753300e+08
Saturday     6.800128e+08
Sunday       6.841189e+08
Thursday     6.760763e+08
Tuesday      6.891058e+08
Wednesday    6.731157e+08
Name: total_amount, dtype: float64

In [27]:
delivered.groupby("order_hour")["order_id"].count()                              # busiest hour


order_hour
0     8382
1     8358
2     8330
3     8382
4     8369
5     8249
6     8438
7     8284
8     8478
9     8314
10    8408
11    8330
12    8385
13    8324
14    8334
15    8307
16    8253
17    8235
18    8330
19    8306
20    8424
21    8270
22    8336
23    8313
Name: order_id, dtype: int64

In [28]:
delivered.groupby("order_hour")["order_id"].nunique()

order_hour
0     8382
1     8358
2     8330
3     8382
4     8369
5     8249
6     8438
7     8284
8     8478
9     8314
10    8408
11    8330
12    8385
13    8324
14    8334
15    8307
16    8253
17    8235
18    8330
19    8306
20    8424
21    8270
22    8336
23    8313
Name: order_id, dtype: int64

In [29]:
   bands = pd.cut(delivered["discount_percent"], [0, 15, 25, 35, 50])
   delivered.groupby(bands, observed=True)["quantity"].mean()

discount_percent
(0, 15]     1.247155
(15, 25]    1.248066
(25, 35]    1.250777
(35, 50]    1.247607
Name: quantity, dtype: float64

In [32]:
import pandas as pd
import os
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass

df = pd.read_csv(os.path.join("data", "customer_clean.csv"))

username = "postgres"
password = getpass("Postgres password: ")
host = "localhost"
port = "5432"
database = "e_commerce"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}"
)

table_name = "customer"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data loaded into '{table_name}' in '{database}'")

Postgres password:  ········


Data loaded into 'customer' in 'e_commerce'


In [33]:
import pandas as pd
import os
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass

df = pd.read_csv(os.path.join("data", "product_clean.csv"))

username = "postgres"
password = getpass("Postgres password: ")
host = "localhost"
port = "5432"
database = "e_commerce"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}"
)

table_name = "product"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data loaded into '{table_name}' in '{database}'")

Postgres password:  ········


Data loaded into 'product' in 'e_commerce'


In [35]:
import pandas as pd
import os
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from getpass import getpass

df = pd.read_csv(os.path.join("data", "sales_clean.csv"))

username = "postgres"
password = getpass("Postgres password: ")
host = "localhost"
port = "5432"
database = "e_commerce"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}"
)

table_name = "sale"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data loaded into '{table_name}' in '{database}'")

Postgres password:  ········


Data loaded into 'sale' in 'e_commerce'


In [36]:
import gzip
import shutil
import os

input_file = os.path.join("data", "sales_clean.csv")
output_file = os.path.join("data", "sales_compressed.csv.gz")

with open(input_file, "rb") as f_in:
    with gzip.open(output_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Original size:", round(os.path.getsize(input_file) / (1024**2), 2), "MB")
print("Compressed size:", round(os.path.getsize(output_file) / (1024**2), 2), "MB")

Original size: 62.67 MB
Compressed size: 13.74 MB


In [37]:
import os

folder = "data"

CUSTOMER_FILE = os.path.join(folder, "customer.csv")
PRODUCT_FILE = os.path.join(folder, "product.csv")
SALES_FILE = os.path.join(folder, "sales_compressed.csv.gz")


In [38]:
import os

print("Current folder:", os.getcwd())
print("Does data folder exist?", os.path.exists("data"))
print("Files in current folder:", os.listdir())

Current folder: C:\Users\sawoo\OneDrive\ドキュメント\Desktop\jupyter
Does data folder exist? True
Files in current folder: ['.ipynb_checkpoints', 'cleaned_data.ipynb', 'customer.ipynb', 'customer_clean.csv', 'data', 'e_com.sql', 'product_clean.csv', 'sales_clean.csv']


In [39]:
import os

print(os.listdir("data"))

['.ipynb_checkpoints', 'customer_clean.csv', 'product_clean.csv', 'sales_clean.csv', 'sales_compressed.csv.gz']


In [15]:
import os

print("Current folder:", os.getcwd())
print("Files inside data folder:", os.listdir("data"))
print("Files in main folder:", os.listdir("."))

Current folder: C:\Users\sawoo\OneDrive\ドキュメント\Desktop\jupyter
Files inside data folder: []
Files in main folder: ['.ipynb_checkpoints', 'cleaned_data.ipynb', 'customer.ipynb', 'customer_clean.csv', 'data', 'e_com.sql', 'product_clean.csv', 'sales_clean.csv']


In [16]:
import os
import shutil

os.makedirs("data", exist_ok=True)

files = [
    "customer_clean.csv",
    "product_clean.csv",
    "sales_clean.csv"
]

for file in files:
    shutil.copy2(file, os.path.join("data", file))

print("Files copied successfully!")
print(os.listdir("data"))

Files copied successfully!
['customer_clean.csv', 'product_clean.csv', 'sales_clean.csv']


In [17]:
import pandas as pd
import os

CUSTOMER_FILE = os.path.join("data", "customer_clean.csv")
PRODUCT_FILE = os.path.join("data", "product_clean.csv")
SALES_FILE = os.path.join("data", "sales_clean.csv")

customer_df = pd.read_csv(CUSTOMER_FILE)
product_df = pd.read_csv(PRODUCT_FILE)
sales_df = pd.read_csv(SALES_FILE)

print("All three cleaned files loaded successfully!")

All three cleaned files loaded successfully!


In [18]:
import os

for file in [
    "data/customer_clean.csv",
    "data/product_clean.csv",
    "data/sales_clean.csv"
]:
    print(file, "->", "Exists" if os.path.exists(file) else "MISSING")

data/customer_clean.csv -> Exists
data/product_clean.csv -> Exists
data/sales_clean.csv -> Exists


In [19]:
print("Customer:", customer_df.shape)
print("Product:", product_df.shape)
print("Sales:", sales_df.shape)

Customer: (40000, 16)
Product: (2000, 13)
Sales: (250000, 32)
